# Exodia Predictor (Streamlined)

This notebook is intentionally minimal:

1. Run the setup cell once (loads `exodia_unified.py`, resolves model weights from `6. Exodia Prediction/Best Models`, and creates `predictor`).
2. Edit user inputs in the run cell (`TEMPERATURE_C`, `SMILES_INPUT`, optional `RUN_SIMILARITY`) and execute.

All property predictions use the best-model checkpoints in `6. Exodia Prediction/Best Models`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd
from IPython.display import display


# Resolve paths robustly whether launched from repo root or Exodia folder.
cwd = Path.cwd().resolve()
if cwd.name == "6. Exodia Prediction":
    EXODIA_DIR = cwd
    REPO_ROOT = cwd.parent
elif (cwd / "6. Exodia Prediction").exists():
    REPO_ROOT = cwd
    EXODIA_DIR = cwd / "6. Exodia Prediction"
else:
    raise FileNotFoundError(
        f"Could not locate '6. Exodia Prediction' from: {cwd}"
    )

DATASETS_DIR = EXODIA_DIR / "Datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)


# Load backend module.
module_path = EXODIA_DIR / "exodia_unified.py"
spec = importlib.util.spec_from_file_location("exodia_unified", module_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load module spec from: {module_path}")
exodia_unified = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = exodia_unified
spec.loader.exec_module(exodia_unified)

build_unified_dataset = exodia_unified.build_unified_dataset
resolve_exodia_best_model_paths = exodia_unified.resolve_exodia_best_model_paths
canonicalize_smiles = exodia_unified.canonicalize_smiles


class ExodiaPredictor:
    def __init__(self, repo_root: Path, exodia_dir: Path):
        self.repo_root = Path(repo_root).resolve()
        self.exodia_dir = Path(exodia_dir).resolve()
        self.datasets_dir = self.exodia_dir / "Datasets"
        self.datasets_dir.mkdir(parents=True, exist_ok=True)
        self.model_paths = resolve_exodia_best_model_paths(self.exodia_dir)
        self._cache: dict[tuple[float, tuple[str, ...]], pd.DataFrame] = {}

    def _cache_key(self, temperature_celsius: float, extra_smiles: list[str] | None) -> tuple[float, tuple[str, ...]]:
        canonical_extra = sorted({s for s in (extra_smiles or []) if s})
        return (float(temperature_celsius), tuple(canonical_extra))

    def _dataset_for_temp(self, temperature_celsius: float, extra_smiles: list[str] | None = None) -> pd.DataFrame:
        key = self._cache_key(temperature_celsius, extra_smiles)
        if key in self._cache:
            return self._cache[key]

        out_path = self.datasets_dir / f"combined_dataset_temp_{float(temperature_celsius):.2f}C.csv"
        df = build_unified_dataset(
            repo_root=self.repo_root,
            output_csv=out_path,
            temperature_c=float(temperature_celsius),
            extra_smiles=extra_smiles,
            model_paths=self.model_paths,
        )
        self._cache[key] = df
        return df

    def predict(self, smiles_input, temperature_celsius: float = 25.0, smiles_column: str = "SMILES") -> pd.DataFrame:
        if isinstance(smiles_input, pd.DataFrame):
            if smiles_column not in smiles_input.columns:
                raise ValueError(f"Column '{smiles_column}' not found in input DataFrame")
            original_df = smiles_input.copy()
            smiles_list = original_df[smiles_column].astype(str).tolist()
        elif isinstance(smiles_input, list):
            original_df = None
            smiles_list = [str(s) for s in smiles_input]
        else:
            raise TypeError("smiles_input must be list[str] or pd.DataFrame")

        canonical = [canonicalize_smiles(s) for s in smiles_list]
        canonical_valid = [s for s in canonical if s is not None]
        dataset = self._dataset_for_temp(temperature_celsius, extra_smiles=canonical_valid)

        keep_cols = [
            "canonical_smiles",
            "BP_value", "BP_source",
            "MP_value", "MP_source",
            "PKA_value", "PKA_source",
            "OP_value", "OP_source",
            "VP_value_log10", "VP_source", "VP_temperature_c",
        ]
        out = dataset[keep_cols].copy()

        query_df = pd.DataFrame({"_query_smiles": smiles_list, "canonical_smiles": canonical})
        merged = query_df.merge(out, on="canonical_smiles", how="left")

        if original_df is not None:
            original_df = original_df.copy()
            original_df["canonical_smiles"] = canonical
            merged = original_df.merge(out, on="canonical_smiles", how="left")

        return merged

    def structural_similarity(
        self,
        query_df: pd.DataFrame,
        temperature_celsius: float,
        dataset_df: pd.DataFrame | None = None,
        top_k: int = 5,
        min_similarity: float = 0.0,
    ) -> dict[str, pd.DataFrame]:
        from rdkit import Chem
        from rdkit.Chem import AllChem, DataStructs

        if dataset_df is None:
            dataset_df = self._dataset_for_temp(temperature_celsius)

        db = dataset_df.dropna(subset=["canonical_smiles"]).drop_duplicates(subset=["canonical_smiles"]).copy()
        db_mols = [Chem.MolFromSmiles(s) for s in db["canonical_smiles"]]
        db_fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) if m is not None else None for m in db_mols]

        if "_query_smiles" in query_df.columns:
            query_smiles = query_df["_query_smiles"].dropna().astype(str).tolist()
        else:
            query_smiles = query_df["canonical_smiles"].dropna().astype(str).tolist()

        results: dict[str, pd.DataFrame] = {}
        for q in query_smiles:
            q_can = canonicalize_smiles(q)
            q_mol = Chem.MolFromSmiles(q_can) if q_can else None
            if q_mol is None:
                results[q] = pd.DataFrame(columns=["canonical_smiles", "similarity"])
                continue

            q_fp = AllChem.GetMorganFingerprintAsBitVect(q_mol, 2, nBits=2048)
            sims = []
            for smi, fp in zip(db["canonical_smiles"], db_fps):
                if fp is None:
                    continue
                sim = DataStructs.TanimotoSimilarity(q_fp, fp)
                if sim >= min_similarity and smi != q_can:
                    sims.append((smi, sim))

            sim_df = pd.DataFrame(sims, columns=["canonical_smiles", "similarity"]).sort_values(
                "similarity", ascending=False
            ).head(top_k)
            if not sim_df.empty:
                sim_df = sim_df.merge(db, on="canonical_smiles", how="left")
            results[q] = sim_df.reset_index(drop=True)

        return results

    def property_similarity(
        self,
        query_df: pd.DataFrame,
        temperature_celsius: float,
        dataset_df: pd.DataFrame | None = None,
        properties=("BP_value", "MP_value", "PKA_value", "OP_value", "VP_value_log10"),
        top_k: int = 5,
    ) -> dict[str, pd.DataFrame]:
        from sklearn.preprocessing import MinMaxScaler

        if dataset_df is None:
            dataset_df = self._dataset_for_temp(temperature_celsius)

        props = [p for p in properties if p in query_df.columns and p in dataset_df.columns]
        if not props:
            raise ValueError("No shared property columns found for comparison")

        ref = dataset_df.dropna(subset=props).copy()
        out: dict[str, pd.DataFrame] = {}

        for _, qrow in query_df.iterrows():
            q_smi = qrow.get("canonical_smiles", None)
            q_vals = pd.DataFrame([qrow[props]], columns=props)
            valid_cols = [c for c in props if pd.notna(q_vals.iloc[0][c])]
            if not valid_cols:
                out[str(q_smi)] = pd.DataFrame(columns=["canonical_smiles", "overall_similarity"])
                continue

            ref_sub = ref.dropna(subset=valid_cols).copy()
            scaler_sub = MinMaxScaler()
            ref_sub_scaled = scaler_sub.fit_transform(ref_sub[valid_cols])
            q_scaled = scaler_sub.transform(q_vals[valid_cols])[0]

            d = np.abs(ref_sub_scaled - q_scaled).mean(axis=1)
            sim = 1.0 - d

            res = ref_sub[["canonical_smiles"] + valid_cols].copy()
            for i, c in enumerate(valid_cols):
                res[f"{c}_similarity"] = 1.0 - np.abs(ref_sub_scaled[:, i] - q_scaled[i])
            res["overall_similarity"] = sim
            res = res.sort_values("overall_similarity", ascending=False).head(top_k).reset_index(drop=True)
            out[str(q_smi)] = res

        return out

    @staticmethod
    def plot_similarity(results: dict[str, pd.DataFrame], similarity_col: str, top_n: int = 3) -> None:
        from rdkit import Chem
        from rdkit.Chem import Draw

        for query, df in results.items():
            if df is None or df.empty:
                print(f"No matches for {query}")
                continue

            top = df.head(top_n)
            q_can = canonicalize_smiles(query)
            q_mol = Chem.MolFromSmiles(q_can) if q_can else None
            if q_mol is None:
                print(f"Invalid query SMILES for plotting: {query}")
                continue

            mols = [q_mol]
            legends = [f"Query\\n{query}"]

            for _, row in top.iterrows():
                mol = Chem.MolFromSmiles(row["canonical_smiles"])
                if mol is not None:
                    mols.append(mol)
                    legends.append(f"{similarity_col}={row[similarity_col]:.2f}\\n{row['canonical_smiles']}")

            img = Draw.MolsToGridImage(
                mols,
                molsPerRow=min(4, len(mols)),
                subImgSize=(260, 220),
                legends=legends,
            )
            display(img)


predictor = ExodiaPredictor(REPO_ROOT, EXODIA_DIR)

print("Repo root:", REPO_ROOT)
print("Exodia dir:", EXODIA_DIR)
print("Datasets dir:", DATASETS_DIR)
print("Best-model checkpoints in use:")
print("  BP :", predictor.model_paths.bp)
print("  MP :", predictor.model_paths.mp)
print("  PKA:", predictor.model_paths.pka)
print("  OP :", predictor.model_paths.op)
print("  VP :", predictor.model_paths.vp)


In [ ]:
# User inputs
TEMPERATURE_C = 25.0
SMILES_INPUT = ["CCO", "c1ccccc1O", "C1=CC(=O)NC=C1"]
RUN_SIMILARITY = False
TOP_K = 5

# Predict
pred_df = predictor.predict(SMILES_INPUT, temperature_celsius=TEMPERATURE_C)
display(pred_df)

# Optional similarity blocks (set RUN_SIMILARITY=True)
if RUN_SIMILARITY:
    dataset_df = predictor._dataset_for_temp(TEMPERATURE_C)

    struct_hits = predictor.structural_similarity(
        query_df=pred_df,
        temperature_celsius=TEMPERATURE_C,
        dataset_df=dataset_df,
        top_k=TOP_K,
    )

    prop_hits = predictor.property_similarity(
        query_df=pred_df.dropna(subset=["canonical_smiles"]),
        temperature_celsius=TEMPERATURE_C,
        dataset_df=dataset_df,
        properties=("BP_value", "MP_value", "PKA_value", "OP_value", "VP_value_log10"),
        top_k=TOP_K,
    )

    print("Top structural neighbors:")
    predictor.plot_similarity(struct_hits, similarity_col="similarity", top_n=min(3, TOP_K))

    print("Top property neighbors:")
    predictor.plot_similarity(prop_hits, similarity_col="overall_similarity", top_n=min(3, TOP_K))
